# 02 MMTIS Loading and Preparation

In this notebook, we load the actual train operation data from the MMTIS dataset.

The goal is to calculate train delays and prepare clean datasets for later visualizations.

## 1. Import libraries

First, we import the libraries that we need for data loading and basic data preparation.

In [1]:
import pandas as pd
from pathlib import Path

## 2. Define data paths

We define the path to the raw MMTIS data and to the folder where we will save processed files.

In [2]:
raw_path = Path("../data/raw/mmtis")
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

## 3. Find all train operation files

The MMTIS folder contains weekly CSV files.  
We use the files named `zugfahrten.csv`, because they contain all train trips, not only delayed trains.

In [3]:
trip_files = sorted(
    raw_path.rglob("*_zugfahrten.csv")
)

len(trip_files)

27

In [4]:
trip_files[:5]

[PosixPath('../data/raw/mmtis/2025/2025-12-01_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-08_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-15_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-22_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-29_mmtis_zugfahrten.csv')]

## 4. Load all train operation files

We load all weekly train operation files and combine them into one dataset.

In [5]:
dfs = []

for file in trip_files:
    df = pd.read_csv(file)
    dfs.append(df)

trips = pd.concat(dfs, ignore_index=True)

In [6]:
trips.shape

(1053324, 9)

In [7]:
trips.head()

,zugnummer,betriebstag,abfahrtzeit_soll,abfahrtzeit_ist,start_betriebsstelle,ankunftzeit_soll,ankunftzeit_ist,ziel_betriebsstelle,fahrzeit_delta
0,20,2025-11-17,2025-11-17T17:15:00+0100,2025-11-17T17:19:13+0100,MLX,NaN,2025-11-17T19:53:37+0100,PAG,NaN
1,21,2025-11-17,2025-11-17T10:21:30+0100,2025-11-17T10:34:31+0100,PAG,2025-11-17T12:44:45+0100,2025-11-17T12:49:46+0100,MLX,-00:08:00
2,22,2025-11-17,2025-11-17T15:15:00+0100,2025-11-17T15:15:42+0100,MLX,NaN,2025-11-17T17:41:51+0100,PAG,NaN
3,23,2025-11-17,2025-11-17T12:22:54+0100,2025-11-17T12:56:25+0100,PAG,2025-11-17T14:44:45+0100,2025-11-17T15:18:47+0100,MLX,00:00:31
4,26,2025-11-17,2025-11-17T11:15:00+0100,2025-11-17T11:19:33+0100,MLX,NaN,2025-11-17T13:37:02+0100,PAG,NaN


## 5. Inspect the dataset

Now we check the columns, data types, and missing values.

In [8]:
trips.columns.tolist()

['zugnummer',
 'betriebstag',
 'abfahrtzeit_soll',
 'abfahrtzeit_ist',
 'start_betriebsstelle',
 'ankunftzeit_soll',
 'ankunftzeit_ist',
 'ziel_betriebsstelle',
 'fahrzeit_delta']

In [9]:
trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1053324 entries, 0 to 1053323
Data columns (total 9 columns):
 #   Column                Non-Null Count    Dtype 
---  ------                --------------    ----- 
 0   zugnummer             1053324 non-null  int64 
 1   betriebstag           1053324 non-null  object
 2   abfahrtzeit_soll      1049472 non-null  object
 3   abfahrtzeit_ist       1047032 non-null  object
 4   start_betriebsstelle  1053324 non-null  object
 5   ankunftzeit_soll      1044231 non-null  object
 6   ankunftzeit_ist       1046983 non-null  object
 7   ziel_betriebsstelle   1053324 non-null  object
 8   fahrzeit_delta        1034256 non-null  object
dtypes: int64(1), object(8)
memory usage: 72.3+ MB


In [10]:
trips.isna().sum()

zugnummer                   0
betriebstag                 0
abfahrtzeit_soll         3852
abfahrtzeit_ist          6292
start_betriebsstelle        0
ankunftzeit_soll         9093
ankunftzeit_ist          6341
ziel_betriebsstelle         0
fahrzeit_delta          19068
dtype: int64

## 6. Convert time columns

The time columns are loaded as text.  
We convert them to datetime format so that we can calculate delays.

In [14]:
time_columns = [
    "abfahrtzeit_soll",
    "abfahrtzeit_ist",
    "ankunftzeit_soll",
    "ankunftzeit_ist"
]

for col in time_columns:
    trips[col] = pd.to_datetime(
        trips[col],
        errors="coerce"
    )

In [15]:
trips[time_columns].dtypes

abfahrtzeit_soll    datetime64[ns, UTC+01:00]
abfahrtzeit_ist     datetime64[ns, UTC+01:00]
ankunftzeit_soll    datetime64[ns, UTC+01:00]
ankunftzeit_ist     datetime64[ns, UTC+01:00]
dtype: object

## 7. Calculate delays

We calculate departure delay and arrival delay in minutes.

A positive value means that the train was late.  
A negative value means that the train was earlier than scheduled.

In [16]:
trips["departure_delay_min"] = (
    trips["abfahrtzeit_ist"] - trips["abfahrtzeit_soll"]
).dt.total_seconds() / 60

In [17]:
trips["arrival_delay_min"] = (
    trips["ankunftzeit_ist"] - trips["ankunftzeit_soll"]
).dt.total_seconds() / 60

In [18]:
trips[[
    "zugnummer",
    "betriebstag",
    "departure_delay_min",
    "arrival_delay_min"
]].head()

,zugnummer,betriebstag,departure_delay_min,arrival_delay_min
0,20,2025-11-17,4.216667,NaN
1,21,2025-11-17,13.016667,5.016667
2,22,2025-11-17,0.700000,NaN
3,23,2025-11-17,33.516667,34.033333
4,26,2025-11-17,4.550000,NaN


## 8. Create time features

We create simple time features from the scheduled departure time.

These features will help us analyze delays by hour, weekday, month, and time of day.

In [19]:
trips["departure_hour"] = trips["abfahrtzeit_soll"].dt.hour
trips["weekday"] = trips["abfahrtzeit_soll"].dt.day_name()
trips["month"] = trips["abfahrtzeit_soll"].dt.month_name()

In [20]:
trips[[
    "abfahrtzeit_soll",
    "departure_hour",
    "weekday",
    "month"
]].head()

,abfahrtzeit_soll,departure_hour,weekday,month
0,2025-11-17 17:15:00+01:00,17.0,Monday,November
1,2025-11-17 10:21:30+01:00,10.0,Monday,November
2,2025-11-17 15:15:00+01:00,15.0,Monday,November
3,2025-11-17 12:22:54+01:00,12.0,Monday,November
4,2025-11-17 11:15:00+01:00,11.0,Monday,November


## 9. Create time-of-day categories

We group the departure hour into simple categories: morning peak, midday, evening peak, and night.

In [21]:
def get_time_of_day(hour):
    if pd.isna(hour):
        return "unknown"
    elif 6 <= hour < 9:
        return "morning_peak"
    elif 9 <= hour < 16:
        return "midday"
    elif 16 <= hour < 19:
        return "evening_peak"
    else:
        return "night"

In [22]:
trips["time_of_day"] = trips["departure_hour"].apply(get_time_of_day)

In [23]:
trips["time_of_day"].value_counts()

time_of_day
unknown         318037
midday          258982
night           225640
morning_peak    126115
evening_peak    124550
Name: count, dtype: int64

## 10. Prepare delay analysis dataset

Some records do not have scheduled or actual arrival times.  
For delay analysis, we keep only records where arrival delay can be calculated.

In [24]:
delay_analysis = trips.dropna(
    subset=["arrival_delay_min"]
).copy()

In [25]:
delay_analysis.shape

(726251, 15)

In [26]:
delay_analysis["arrival_delay_min"].describe()

count    726251.000000
mean          1.993230
std           7.176926
min        -560.483333
25%          -0.183333
50%           0.450000
75%           2.066667
max         782.733333
Name: arrival_delay_min, dtype: float64

## 11. Handle extreme delay values

Some delay values are very large or very negative.  
For visualizations, we create a capped delay column.

This keeps the main patterns visible and reduces the effect of extreme outliers.

In [27]:
delay_analysis["arrival_delay_capped"] = (
    delay_analysis["arrival_delay_min"]
    .clip(lower=-30, upper=120)
)

In [28]:
delay_analysis[[
    "arrival_delay_min",
    "arrival_delay_capped"
]].describe()

,arrival_delay_min,arrival_delay_capped
count,726251.000000,726251.000000
mean,1.993230,1.994114
std,7.176926,6.378355
min,-560.483333,-30.000000
25%,-0.183333,-0.183333
50%,0.450000,0.450000
75%,2.066667,2.066667
max,782.733333,120.000000


## 12. Check punctuality

We define a train as on time if it arrives no more than 5 minutes late.

In [29]:
delay_analysis["on_time"] = (
    delay_analysis["arrival_delay_min"] <= 5
)

In [30]:
on_time_rate = delay_analysis["on_time"].mean()

on_time_rate

np.float64(0.898782927665504)

## 13. Create summary table by hour

This table shows the average arrival delay for each departure hour.

In [31]:
hour_summary = (
    delay_analysis
    .groupby("departure_hour", as_index=False)
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

hour_summary

,departure_hour,mean_delay,median_delay,number_of_trips
0,0.0,1.608176,0.050000,9620
1,1.0,1.086015,-0.100000,2343
2,2.0,1.459279,0.083333,1383
3,3.0,2.771694,0.166667,2997
4,4.0,1.254605,0.266667,22593
5,5.0,1.738155,0.416667,41973
6,6.0,2.124502,0.600000,47695
7,7.0,2.288440,0.550000,40174
8,8.0,2.186856,0.441667,35898
9,9.0,1.908482,0.350000,35134


## 14. Create summary table by weekday

This table shows the average arrival delay for each weekday.

In [32]:
weekday_summary = (
    delay_analysis
    .groupby("weekday", as_index=False)
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

weekday_summary

,weekday,mean_delay,median_delay,number_of_trips
0,Friday,2.383747,0.600000,109559
1,Monday,2.093075,0.550000,108853
2,Saturday,1.350812,0.200000,93061
3,Sunday,1.303620,0.133333,84943
4,Thursday,2.188361,0.583333,108209
5,Tuesday,2.266212,0.566667,108937
6,Wednesday,2.120418,0.566667,110091


## 15. Create summary table by station pair

This table shows average delay between start and destination operating points.

In [33]:
station_pair_summary = (
    delay_analysis
    .groupby(
        ["start_betriebsstelle", "ziel_betriebsstelle"],
        as_index=False
    )
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

In [34]:
station_pair_summary.sort_values(
    "mean_delay",
    ascending=False
).head(20)

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
1400,JB,LN,120.0,120.0,1
644,FB,GZS,120.0,120.0,1
1159,HE,HW H1,120.0,120.0,1
2988,RO H2,STTS12,120.0,120.0,1
1289,HP U1,MTH,120.0,120.0,1
2830,PG,PG G,120.0,120.0,1
2796,PAG,PW,120.0,120.0,2
2791,PAG,LZW,120.0,120.0,1
2790,PAG,HEZ,120.0,120.0,1
1445,K S11,I,120.0,120.0,1


## 16. Save processed datasets

We save the prepared trip-level data and the summary tables.  
These files will be used later for visualizations and the dashboard.

In [35]:
delay_analysis.to_csv(
    processed_path / "mmtis_delay_analysis.csv",
    index=False
)

hour_summary.to_csv(
    processed_path / "mmtis_hour_summary.csv",
    index=False
)

weekday_summary.to_csv(
    processed_path / "mmtis_weekday_summary.csv",
    index=False
)

station_pair_summary.to_csv(
    processed_path / "mmtis_station_pair_summary.csv",
    index=False
)

In [36]:
print("Saved files:")
print(processed_path / "mmtis_delay_analysis.csv")
print(processed_path / "mmtis_hour_summary.csv")
print(processed_path / "mmtis_weekday_summary.csv")
print(processed_path / "mmtis_station_pair_summary.csv")

Saved files:
../data/processed/mmtis_delay_analysis.csv
../data/processed/mmtis_hour_summary.csv
../data/processed/mmtis_weekday_summary.csv
../data/processed/mmtis_station_pair_summary.csv
